# 02C Data Cleaning and Feature Engineering

> 🟢 **Level A · Required**

Goal: `raw table → audit → clean → construct features → select columns → ML-ready table`. Use the public COFSpace CoRE-COF CO₂ 1 bar dataset.


In [ ]:
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv')
display(df.head())
display(pd.DataFrame({'dtype':df.dtypes.astype(str),'missing':df.isna().sum(),'missing_%':(100*df.isna().mean()).round(2),'unique':df.nunique()}))


In [ ]:
target='CO2-1 bar (mol/kg)'
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
work=df[features+[target]].copy()
work['LCD_PLD_ratio']=work['LCD (Å)']/work['PLD (Å)']
work['heteroatom_%']=work[['%N','%O','%Metalloid','%Halogen','%Ametal']].sum(axis=1)
display(work.head())


Missing does not mean zero. Scaling is important for some models but not usually required for trees. Remove target-derived variables, IDs, unavailable features and duplicates before sophisticated selection.

### Completion criterion
Produce explicit `X` and `y` and document feature source, units and preprocessing.


## Turn the audit into an explicit cleaning decision
Keep source row indices for traceability; they are not verified material IDs. Missing targets cannot be imputed. Convert numeric columns explicitly, log rejected rows, and treat nonpositive PLD as invalid for a ratio. Fit median imputation and scaling only inside training folds (03B).


In [ ]:
import numpy as np
numeric = df[features + [target]].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
rejected = df.loc[numeric[target].isna()].copy()
clean = numeric.loc[numeric[target].notna()].copy()
clean['LCD_PLD_ratio'] = clean['LCD (Å)'] / clean['PLD (Å)'].where(clean['PLD (Å)'] > 0)
clean['heteroatom_%'] = clean[['%N','%O','%Metalloid','%Halogen','%Ametal']].sum(axis=1, min_count=5)
X = clean.drop(columns=target)
y = clean[target]
print('kept / missing target:', len(clean), len(rejected))
print('duplicate feature rows (investigate; do not silently drop):', X.duplicated().sum())
display(X.isna().sum().to_frame('missing'))


## Exercise and handoff
Produce a feature dictionary (name, unit, source, availability at prediction time). Explain why zero is not a universal missing-value replacement. Keep an exclusion log and pass X/y to 03A; pass the unfitted preprocessing pipeline to 03B.


## Sources and further reading
[Dataset contracts / 数据使用约定](../../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
